# WAQ at scale (Numba) — paper simulations + large \(n\)

This follows **Section 5 (Simulations)** of Athey, Bickel, Chen, Imbens & Pollmann, *JRSS-B* 2023 ([DOI](https://doi.org/10.1093/jrsssb/qkad072), [arXiv:2109.02603](https://arxiv.org/abs/2109.02603)).

**Paper setup (known distributions)**

- \(n=20{,}000\): **10,000 treated + 10,000 control**.
- **True treatment effect = 0** (same potential-outcome law in both arms).
- Distributions: **standard Normal**, **Double Exponential (Laplace)**, **Cauchy**.
- **Cauchy** is the dramatic case: the **difference in means** has enormous variance (and huge RMSE in finite samples); **WAQ** stays close to the semiparametric bound (**Table 1**, Cauchy block).

**Why your small tutorial can favor OLS/DiM**

- Light-tailed or moderate-\(n\) settings look like **Normal** in Table 1: WAQ and DiM are similar.
- **Bootstrap SE** for WAQ is noisy at small \(n\); DiM gets a tidy closed form.
- **Do not** use a tiny `density_sample_size` (e.g. 10k from millions of controls): KDE weights must track the **control** density — subsampling too aggressively biases WAQ (the paper uses full-sample density in Table 1; their footnote allows a **large** subsample for very big \(n\)).

**Requirements:** Numba — in **zsh** quote extras: `pip install 'cluster-experiments[performance]'` or `pip install numba`.


In [ ]:
import time
import numpy as np

from cluster_experiments.waq.estimator import estimate_waq_tau
from cluster_experiments.waq.kde import get_numba_kde

if get_numba_kde() is None:
    raise RuntimeError(
        "Install Numba (zsh: quote extras): pip install 'cluster-experiments[performance]' or pip install numba"
    )

N_PER_ARM_PAPER = 10_000  # 10k + 10k as in Table 1
TRUE_TAU = 0.0
RNG = np.random.default_rng(2026)


## DGPs (match paper §5.1)

- **Normal:** \(Y \sim \mathcal{N}(0,1)\).
- **Laplace:** unit variance, \(b = 1/\sqrt{2}\) so \(\mathrm{Var}(Y)=1\).
- **Cauchy:** standard Cauchy (no finite mean/variance in the population; DiM is unstable in finite samples).


In [ ]:
def sample_control_treat(n, dist, rng, tau=0.0):
    """Independent arms, same law + location shift tau on treatment (paper: tau=0)."""
    if dist == "normal":
        y0 = rng.standard_normal(n)
        y1 = rng.standard_normal(n) + tau
    elif dist == "laplace":
        b = 1.0 / np.sqrt(2.0)
        y0 = rng.laplace(0.0, b, size=n)
        y1 = rng.laplace(0.0, b, size=n) + tau
    elif dist == "cauchy":
        y0 = rng.standard_cauchy(n)
        y1 = rng.standard_cauchy(n) + tau
    else:
        raise ValueError(dist)
    return y0, y1


## Monte Carlo — Cauchy (Table 1, bottom block)

Paper (10,001 replications): diff. in means **SD ≈ 127**, **RMSE ≈ 127**; **waq** **SD ≈ 0.021**, **RMSE ≈ 0.021**.

Below uses **800** reps (several minutes with Numba); set `n_mc=150` for a quick smoke test, or `2000+` to get closer to the published SDs.


In [ ]:
def mc_paper(dist, n_mc=800, n_per_arm=N_PER_ARM_PAPER, tau=TRUE_TAU, seed=0, density_cap=None):
    """
    density_cap: None => all controls for KDE (paper Table 1).
    """
    dims, waqs = [], []
    for i in range(n_mc):
        r = np.random.default_rng(seed + i)
        c, t = sample_control_treat(n_per_arm, dist, r, tau=tau)
        dims.append(float(t.mean() - c.mean()))
        kwargs = dict(use_numba=True, rng=r)
        if density_cap is not None and len(c) > density_cap:
            kwargs["density_sample_size"] = density_cap
        tw, _ = estimate_waq_tau(c, t, **kwargs)
        waqs.append(tw)
    return np.array(dims), np.array(waqs)


t0 = time.perf_counter()
dim_c, waq_c = mc_paper("cauchy", n_mc=800, density_cap=None)
print(f"Cauchy MC (800 reps, 10k/10k, full control for KDE): {time.perf_counter() - t0:.1f}s\n")
for name, x in [("diff. in means", dim_c), ("waq (this package)", waq_c)]:
    bias = x.mean() - TRUE_TAU
    sd = x.std(ddof=1)
    rmse = np.sqrt(np.mean((x - TRUE_TAU) ** 2))
    print(f"{name:22s}  bias={bias:8.4f}  SD={sd:10.4f}  RMSE={rmse:10.4f}")
print("\nPaper Table 1 (Cauchy): diff. in means  SD≈127 RMSE≈127;  waq  SD≈0.021 RMSE≈0.021")


## Normal & Laplace — WAQ ≈ DiM (Table 1, top/middle)

Under Normal, optimal WAQ weights are nearly constant; under Laplace, DiM is already inefficient but not catastrophic like Cauchy.


In [ ]:
for dist, label in [("normal", "Normal"), ("laplace", "Laplace")]:
    d, w = mc_paper(dist, n_mc=400, seed=10_000)
    print(
        f"{label}: DiM RMSE={np.sqrt(np.mean(d**2)):.5f}  WAQ RMSE={np.sqrt(np.mean(w**2)):.5f}"
    )


## Large \(n\) timing (Cauchy, subsample for KDE)

Paper footnote: with very large samples, density can be fit on a **random subsample** of controls. Here we draw **many users per arm** and cap KDE to `DENSITY_SUBSAMPLE` (increase if you have RAM/time). **Single draws** of DiM on Cauchy are not meaningful; the MC above is the right comparison.


In [ ]:
# 5M+5M default; set to 1_000_000 or lower on laptops
N_BIG = 5_000_000
DENSITY_SUBSAMPLE = 500_000

r = np.random.default_rng(99)
y0 = r.standard_cauchy(N_BIG)
y1 = r.standard_cauchy(N_BIG)
print(f"Draw {N_BIG:,} × 2 Cauchy outcomes…")

warm_c, warm_t = sample_control_treat(8000, "cauchy", np.random.default_rng(0))
_, _ = estimate_waq_tau(warm_c, warm_t, use_numba=True, rng=np.random.default_rng(0))

t0 = time.perf_counter()
tau_waq, info = estimate_waq_tau(
    y0,
    y1,
    density_sample_size=DENSITY_SUBSAMPLE,
    use_numba=True,
    rng=np.random.default_rng(42),
)
t_waq = time.perf_counter() - t0
t0 = time.perf_counter()
dim_big = float(y1.mean() - y0.mean())
t_dim = time.perf_counter() - t0

print(f"DiM (one draw): {dim_big:.4f}  [{t_dim:.3f}s]  ← ignore magnitude; Cauchy mean is undefined")
print(f"WAQ τ̂:          {tau_waq:.4f}  [{t_waq:.2f}s]  info {info}")
print("(Truth τ=0; WAQ targets the common QTE, which is 0 under this DGP.)")


### Reference — efficient weights on Cauchy (paper Fig. 2)

On quantile \(u\): \(w(u) \propto -\cos(2\pi u)\sin^2(\pi u)\): mass near the **median**, negative weights in the tails — intuition for why **averaging raw outcomes** (DiM) is disastrous while **reweighted quantile contrasts** (WAQ) remain stable.


In [ ]:
try:
    import matplotlib.pyplot as plt

    u = np.linspace(0.001, 0.999, 500)
    w = -np.cos(2 * np.pi * u) * (np.sin(np.pi * u) ** 2)
    w = w / w.mean()
    plt.figure(figsize=(7, 3))
    plt.plot(u, w)
    plt.xlabel("quantile u")
    plt.ylabel("weight (mean 1)")
    plt.title("Paper Fig. 2: Cauchy efficient weights vs u")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("Install matplotlib to plot the weight curve.")
